---
title: "04. Results DB & batch workflows"
description: "The generic results database every workflow reports into, batch fan-out as parent/child rows written by a one-shot Compose container, and a stateless continuation rule instead of an orchestration engine."
---

This chapter builds the platform's operational backbone: one generic results DB
that records the state of every run of every type, plus the first workflow that
needs it in earnest. Batch inference fans out as **one parent row per batch and
one child row per chunk**, tracks per-chunk success/failure, distinguishes
transient from permanent failures, and can *run until done* without any
orchestration engine, because all the state it needs lives in the results DB.

On the Compose stack the consumer is a batch scoring job: a one-shot container
built from `src/batch_job/Dockerfile`, scoring model versions registered by
[chapter 03](03-reproducible-training.ipynb). Triggers here are **on-demand**
only, via the dashboard buttons and runner API of
[chapter 02](02-local-foundation.ipynb); scheduled execution arrives when Part II
runs this same image as an ACA Job with cron triggers
([chapter 11](11-porting-to-aca.ipynb)).


## Design — the generic results DB

One Postgres table records the state of every job of every type:

```sql
CREATE TABLE results (
    id           TEXT PRIMARY KEY,                 -- UUID, or deterministic hash for idempotent items
    parent_id    TEXT NULL REFERENCES results(id), -- NULL = top-level run; set = child
    name         TEXT NOT NULL,                    -- workflow/task type, e.g. 'batch:score-fraud'
    status       TEXT NOT NULL,                    -- PENDING|STARTED|SUCCESS|RETRY|FAILURE|REVOKED
    output       JSONB NULL,                       -- per-task metadata; big payloads go to object storage
    error        TEXT NULL,
    attempts     INT NOT NULL DEFAULT 0,
    triggered_by TEXT NOT NULL,                    -- 'schedule' or caller identity (audit)
    created_at   TIMESTAMPTZ NOT NULL DEFAULT now(),
    updated_at   TIMESTAMPTZ NOT NULL DEFAULT now()
);
CREATE INDEX ON results (parent_id, status);
```

`RETRY` means transient/retriable; `FAILURE` means permanent. Batch inference uses
**one parent row per batch and one child row per item/chunk**, giving per-item
success/failure without any bespoke ledger. Locally the table lives in the
`results` database inside the Compose Postgres container
([chapter 02](02-local-foundation.ipynb)); in Part II it moves to Azure Database
for PostgreSQL unchanged.

## Design — triggers and the continuation rule

Three ways a workflow can start:

- **On-demand** — the dashboard's trigger buttons or a direct call to the runner
  API starts the one-shot batch container; every row it writes records the
  caller in `triggered_by`.
- **Scheduled (Part II)** — the same image runs as an ACA Job whose definition
  carries a cron expression; each tick starts a fresh execution with
  `triggered_by='schedule'`, with no scheduler process to keep alive
  ([chapter 11](11-porting-to-aca.ipynb)).
- **Event** — started from an event source when a use case needs it (deferred;
  see the extensions at the end of this chapter).

A linear pipeline (extract → validate → score → publish) is one script in one
container. *Run until done* is a stateless rule over the results DB:
**re-dispatch children still in `PENDING`/`RETRY` up to the `attempts` cap; the
batch is done when no child is non-terminal.** Any process that can query
Postgres can apply the rule, so a crashed run recovers by simply starting again.


## Build in `projects/ml-platform/`

```
projects/ml-platform/
├── src/ml_platform/results/
│   ├── schema.sql              # results DDL, owned once: CREATE TABLE IF NOT EXISTS + index
│   ├── store.py                # create_run, create_children, mark, pending_children, finalize_parent
│   └── continuation.py         # run_until_done: PENDING/RETRY loop + circuit breaker
├── src/batch_job/
│   ├── Dockerfile              # pinned base + deps + ml_platform package (same pattern as train_job)
│   ├── requirements.txt        # mlflow/sklearn/pandas pinned + psycopg (+ azure-identity for Part II auth)
│   └── score.py                # entrypoint: load model → create parent/children → run_until_done
└── demo/
    ├── docker-compose.yml      # + one-shot batch service; BATCH_CHUNK_SIZE pinned to 100
    └── postgres/init/01-create-results.sql  # creates the results db and applies the DDL at first boot
```

The `results/` module is environment-neutral. It connects through ordinary
`PG*` variables, so the same code writes to the Compose Postgres container today
and to Azure Database for PostgreSQL in Part II; only connection settings differ
(local password auth versus short-lived Entra tokens from a managed identity).
The DDL is owned once by `schema.sql`: the Compose init script materializes the
identical table definition when Postgres first initializes its volume, and the
Part II deploy scripts apply the same file after `grants.sql`, so both
environments converge on one schema.

This module formalizes the table contract that `common/results.py` began
writing in [chapter 03](03-reproducible-training.ipynb): the `record_run`
context manager delegates to this store, so training rows and batch rows share
one writer, one status vocabulary, and one table from here on.


## How the pieces connect

### Results module (`ml_platform/results/`)

`schema.sql` owns the DDL once, and every environment applies it the same way.
Locally, `demo/postgres/init/01-create-results.sql` creates the `results`
database and executes the identical `CREATE TABLE IF NOT EXISTS` statements;
the Postgres image runs init scripts only when its data volume is empty, so a
`docker compose down -v` reset rebuilds the schema for free. In Part II the
deploy scripts apply `schema.sql` itself after `grants.sql`, the same pattern
as the Postgres principals setup.

`store.py` exposes five functions that cover the full lifecycle:

| Function | Purpose |
|---|---|
| `create_run(name, triggered_by=…)` | Insert a top-level row in PENDING |
| `create_children(parent_id, items, name=…, triggered_by=…)` | Bulk-insert child rows with **deterministic ids** (`SHA-256(name:item_key)[:32]`) — idempotent, `ON CONFLICT DO NOTHING` |
| `mark(run_id, status, output=…, error=…, increment_attempts=…)` | Update a row's status and metadata |
| `pending_children(parent_id, max_attempts=…)` | Query PENDING/RETRY children below the attempt cap |
| `finalize_parent(parent_id)` | Set parent SUCCESS (all children terminal, none FAILURE) or FAILURE |

All five are no-ops when `PGHOST` is unset, so any job linked against the store
stays runnable outside a live database (tests, ad-hoc experiments).

### Continuation rule (`ml_platform/results/continuation.py`)

`run_until_done(parent_id, processor, max_attempts=3, max_iterations=10)`
applies the rule in a loop:

1. Fetch `pending_children`; each attempt increments the child's `attempts`.
2. For each child: call `processor(child)`, mark SUCCESS; on `BatchItemFailure`
   mark FAILURE (permanent); on any other exception mark RETRY (transient).
3. If no child changed state between iterations (no progress), circuit-break →
   mark the parent FAILURE.
4. Stop when no eligible children remain; call `finalize_parent`.

A crashed or re-deployed job simply re-evaluates `pending_children` and
continues from where the DB says it stopped; there is no in-memory queue to
rebuild.

### Batch scoring (`src/batch_job/score.py`)

The entrypoint is symmetric to `train.py` but read-only to MLflow:

1. Load a pinned model: `--model-version` names any version produced by the
   chapter 03 pipeline; omitted, it falls back to a registry alias. Which
   version *should* be scored is promotion policy, covered in
   [chapter 05](05-online-serving.ipynb) and [chapter 08](08-environment-contract.ipynb).
2. Read the input CSV (`--data-source`) and split into chunks of `--chunk-size`
   rows, defaulted from `BATCH_CHUNK_SIZE` (100, pinned in
   `demo/docker-compose.yml`).
3. Create one parent row plus one child row per chunk via
   `store.create_children`.
4. Call `run_until_done`; the processor scores one chunk and calls `store.mark`
   with the per-chunk output summary.
5. Record parent output (`n_chunks`, `total_rows`, `model_ref`) and exit
   non-zero on anything other than SUCCESS.

Remaining flags follow the same shape: delimiter and target column for the CSV,
a retry cap for the continuation rule, and `--triggered-by` for the audit
column.

### Trigger path (local)

In the Compose stack the job starts two ways. At stack startup the one-shot
`batch` service waits for `train` to complete (so model version 1 exists) and
scores the sample CSV once. After that, everything is on-demand: the dashboard's
**Run batch scoring** button and `POST /api/runs/batch/trigger` reach the
runner, which validates parameters (`model_version`, `chunk_size`,
`max_attempts`), launches `score.py` as a subprocess, and returns an execution
id immediately while the job runs in the background. The runner passes its
execution name to the job as `RESULTS_RUN_ID`, so the response id *is* the
parent results row id: one identifier follows the run from trigger response to
`GET /api/results/{id}`.

**Part II preview.** The same image ships unchanged as the `batch` ACA Job: the
Terraform module mirrors `train_job`, adds a manual trigger alongside a dynamic
schedule trigger gated on a cron variable, and runs under the `id-jobs-batch`
managed identity provisioned with the foundation
([chapter 10](10-azure-foundation.ipynb)). [Chapter 11](11-porting-to-aca.ipynb)
covers the port, where scheduling becomes real and the runner retires exactly as
[chapter 02](02-local-foundation.ipynb) promised.


## Golden-path position & acceptance evidence

This chapter adds the `registered version → batch scoring → per-chunk results`
segment of the golden path and builds the operational backbone that every later
chapter reports into.

**Acceptance evidence:**

- One query returns full status/output/error for a run **and its children**
  (`psql` against the local `results` database, or the dashboard's results API).
- A batch of N chunks yields one parent row + N child rows; forcing a transient
  failure on one chunk marks it `RETRY`, and the continuation rule re-dispatches
  only that chunk until it reaches a terminal state.
- An on-demand trigger records its caller in `triggered_by`, and the trigger
  response's execution id is the parent row id, so one identifier follows the
  run from `POST /trigger` to `GET /api/results/{id}`.
- Scheduling is deliberately out of scope for the local POC: cron execution is
  exercised in [Part II](10-azure-foundation.ipynb) when ACA Job schedule
  triggers start this same image ([chapter 11](11-porting-to-aca.ipynb)).


## Extensions (deferred from the MVP)

| Deferred | Contract | MVP substitute |
|---|---|---|
| Broker / queue backpressure for huge fan-out (revisit when a single-container fan-out saturates) | `docs/04` | Parent/child rows + continuation (see [chapter 15](15-broker-upgrade.ipynb)) |
| Event-triggered workflows (revisit when a use case needs reactive starts) | `docs/04` | On-demand triggers now; cron via ACA Job schedule triggers in Part II ([chapter 11](11-porting-to-aca.ipynb)) |
| Cross-workflow DAGs (revisit when pipelines need inter-job dependencies) | `docs/04` | One script per job; no orchestration graph |

Next: **[05 — Online serving & promotion](./05-online-serving.ipynb)** puts a model
version behind an HTTP endpoint with version-based promotion and rollback.
